<a href="https://colab.research.google.com/github/Spurthimulkanuri/ServiceHiveAssignment/blob/main/ServiceHiveAssignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# -------- INSTALL (run once) --------
!pip install -q langchain langchain-community faiss-cpu sentence-transformers

# -------- IMPORTS --------
import re
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import CharacterTextSplitter

# -------- EMAIL VALIDATION --------
def is_valid_email(email):
    pattern = r'^[\w\.-]+@([\w\.-]+)\.\w+$'
    match = re.match(pattern, email)

    if not match:
        return False

    domain = email.split("@")[1].lower()

    valid_domains = ["gmail.com", "yahoo.com", "outlook.com", "hotmail.com"]

    return domain in valid_domains

# -------- CREATE KNOWLEDGE FILE --------
with open("knowledge.md", "w") as f:
    f.write("""
AutoStream Pricing:

Basic Plan:
$29/month
10 videos/month
720p resolution

Pro Plan:
$79/month
Unlimited videos
4K resolution
AI captions

Policies:
No refunds after 7 days
24/7 support only for Pro plan
""")

# -------- INTENT DETECTION --------
def detect_intent(text):
    text = text.lower()

    if any(word in text for word in ["buy", "start", "try", "subscribe", "purchase", "want"]):
        return "high_intent"
    elif any(word in text for word in ["price", "pricing", "plan", "plans", "cost"]):
        return "pricing"
    elif any(word in text for word in ["hi", "hello"]):
        return "greeting"

    return "general"

# -------- LEAD TOOL --------
def mock_lead_capture(name, email, platform):
    print(f"Lead captured: {name}, {email}, {platform}")

# -------- RAG SETUP --------
loader = TextLoader("knowledge.md")
docs = loader.load()

splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20)
docs = splitter.split_documents(docs)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db = FAISS.from_documents(docs, embeddings)

def retrieve(query):
    return db.similarity_search(query, k=2)

# -------- RESPONSE --------
def generate_response(query, docs):
    context = "\n".join([d.page_content for d in docs])
    return f"""Here are our plans:

{context}

Let me know if you'd like to get started!
"""

# -------- STATE --------
state = {
    "messages": [],
    "intent": "",
    "name": None,
    "email": None,
    "platform": None
}

# -------- CHAT FUNCTION --------
def chat(msg):
    state["messages"].append(msg)

    # 🔥 Lead flow
    if state.get("intent") == "high_intent":

        if not state.get("name"):
            state["name"] = msg
            return "Please enter your email"

        elif not state.get("email"):
            # ✅ FIXED VALIDATION (INDENT CORRECT)
            if not is_valid_email(msg):
                return "❌ Please enter a valid email (example: name@gmail.com)"

            state["email"] = msg
            return "Which platform do you use? (YouTube/Instagram)"

        elif not state.get("platform"):
            state["platform"] = msg
            mock_lead_capture(state["name"], state["email"], state["platform"])
            return "🎉 Lead captured successfully!"

    # 🔥 Normal flow
    intent = detect_intent(msg)

    if intent == "greeting":
        return "Hi! 👋 How can I help you with AutoStream?"

    elif intent == "pricing":
        docs = retrieve(msg)
        return generate_response(msg, docs)

    elif intent == "high_intent":
        state["intent"] = "high_intent"
        state["name"] = ""
        state["email"] = ""
        state["platform"] = ""
        return "Great! Let's get started. What's your name?"

    return "I'm here to help! 😊"

/tmp/ipykernel_9925/37816099.py:70: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:

while True:
    user_input = input("You: ")

    if user_input.lower() == "exit":
        print("Bot: Bye 👋")
        break

    response = chat(user_input)
    print("Bot:", response)

You: Hi
Bot: Hi! 👋 How can I help you with AutoStream?
You: What is pricing
Bot: Here are our plans:

AutoStream Pricing:

Basic Plan:
$29/month
10 videos/month
720p resolution

Pro Plan:
$79/month
Unlimited videos
4K resolution
AI captions
Policies:
No refunds after 7 days
24/7 support only for Pro plan

Let me know if you'd like to get started!

You: I want pro plan
Bot: Great! Let's get started. What's your name?
You: Spurthi
Bot: Please enter your email
You: Spurth@12
Bot: ❌ Please enter a valid email (example: name@gmail.com)
You: spurthimulkanuri@gmail.com
Bot: Which platform do you use? (YouTube/Instagram)
You: Youtube
Lead captured: Spurthi, spurthimulkanuri@gmail.com, Youtube
Bot: 🎉 Lead captured successfully!
